In [1]:
# --- Package import ---

import pandas as pd
import folium
from folium import plugins
import numpy as np
from scipy.spatial.distance import cdist

import calliope

from ruamel.yaml import YAML
import pyrosm

import geopandas as gpd
from shapely.geometry import Point, LineString

from owslib.wfs import WebFeatureService
import io

import requests

In [2]:
# --- Haversine distance function ---

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance in kilometers between two points 
    on the earth (specified in decimal degrees).
    """
    # Convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    # Haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    # Radius of earth in kilometers. Use 3956 for miles
    r = 6371 
    return c * r

In [3]:
# --- API calls to obtain geodata ---

# Fetch building GeoJSON data for area 4011 from the TNO Warmteprofielgenerator API (may not be legal)
geojson_url = "https://hlc-api.warmteprofielengenerator.nl/building_data/geojson/4011"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:145.0) Gecko/20100101 Firefox/145.0",
    "Accept": "application/json",
    "Referer": "https://www.warmteprofielengenerator.nl/",
    "Origin": "https://www.warmteprofielengenerator.nl"
}

response = requests.get(geojson_url, headers=headers)
response.raise_for_status()
data = response.json()

# Extract properties from each feature
if "features" in data and len(data["features"]) > 0:
    properties_list = [f.get("properties", {}) for f in data["features"]]
    residential_heat_demand = pd.DataFrame(properties_list)
else:
    print("No features to export.")

# Extract only the heat demand data from the InfluxDB/Grafana API
url = (
    "https://hlc-grafana.warmteprofielengenerator.nl/api/datasources/proxy/2/query?db=tnohlc"
    "&q=SELECT%20sum(%22P_heat%22)%20FROM%20%22tot-4011%22%20WHERE%20time%20%3E%3D%201546300800000ms%20and%20time%20%3C%3D%201577836799999ms%20GROUP%20BY%20time(1h)%20fill(null)"
    "&epoch=ms" )

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:145.0) Gecko/20100101 Firefox/145.0",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Referer": "https://hlc-grafana.warmteprofielengenerator.nl/d-solo/BJBoKWivz/warmtevraagprofiel-2019-4011?orgId=1&panelId=1&from=1546300800000&to=1577836799999&theme=light",
    "x-grafana-org-id": "1",
    "DNT": "1",
    "Connection": "keep-alive",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin",
    "Priority": "u=4",
    "TE": "trailers"
}

response = requests.get(url, headers=headers)
response.raise_for_status()
data = response.json()

# Query BAG buildings (pand) from PDOK WFS API as GeoJSON

# Define your bounding box (minx, miny, maxx, maxy) in EPSG:4326 (WGS84)
bbox = [4.354481303046308, 51.990211796688996, 4.36349681889779, 51.99831039946793]

# WFS API returns maxFeatures=1000 by default; we need to paginate to get more
def fetch_all_features(bbox, max_features=1000):
    features = []
    start_index = 0
    while True:
        wfs_url = (
            "https://service.pdok.nl/lv/bag/wfs/v2_0?"
            "service=WFS&version=1.1.0&request=GetFeature"
            "&typeName=bag:pand"
            "&outputFormat=application/json"
            f"&bbox={bbox[1]},{bbox[0]},{bbox[3]},{bbox[2]},urn:ogc:def:crs:EPSG::4326"
            f"&maxFeatures={max_features}"
            f"&startIndex={start_index}"
        )
        response = requests.get(wfs_url)
        response.raise_for_status()
        gdf = gpd.read_file(io.BytesIO(response.content))
        if gdf.empty:
            break
        features.append(gdf)
        if len(gdf) < max_features:
            break
        start_index += max_features
    if features:
        return gpd.GeoDataFrame(pd.concat(features, ignore_index=True), crs=features[0].crs)
    else:
        return gpd.GeoDataFrame()

gdf = fetch_all_features(bbox)

In [4]:
# --- Demand node definition ---

# Extract the heat demand series only
series = data.get("results", [])[0].get("series", []) if "results" in data else []
heat_df = None
for s in series:
    name = s.get("name", "")
    columns = s.get("columns", [])
    values = s.get("values", [])
    if name == "tot-4011":
        heat_df = pd.DataFrame(values, columns=columns)

# --- New calculation as requested ---
if (residential_heat_demand is not None) and (heat_df is not None):
    warmtevraag_sum = residential_heat_demand["Warmtevraag"].sum()
    # Calculate share for each building
    residential_heat_demand["share"] = residential_heat_demand["Warmtevraag"] / warmtevraag_sum
    # Get max hourly heat demand from heat_df
    max_heat = heat_df["sum"].max()
    # Multiply share by max_heat
    residential_heat_demand["Peak heat demand (kW)"] = residential_heat_demand["share"] * max_heat/1000
else:
    print("Data not available for calculation.")

# Reproject to a projected CRS for centroid calculation, then convert centroids back to WGS84 for lon/lat
if gdf.crs is not None and gdf.crs.to_epsg() == 4326:
    gdf_proj = gdf.to_crs(epsg=28992)  # Projected CRS for NL
    centroids_proj = gdf_proj.centroid
    centroids_wgs = gpd.GeoSeries(centroids_proj, crs=28992).to_crs(epsg=4326)
    gdf['lon'] = centroids_wgs.x
    gdf['lat'] = centroids_wgs.y
else:
    # Fallback: use centroid for non-point geometries, but ensure output is in WGS84
    centroids = gdf.geometry.centroid
    centroids_wgs = gpd.GeoSeries(centroids, crs=gdf.crs).to_crs(epsg=4326)
    gdf['lon'] = centroids_wgs.x
    gdf['lat'] = centroids_wgs.y

# Prepare export DataFrame with identification and coordinates
export_cols = ['identificatie', 'lon', 'lat']
export_df = gdf[export_cols].copy() if all(col in gdf.columns for col in export_cols) else gdf[[c for c in export_cols if c in gdf.columns]].copy()
export_df = export_df.rename(columns={'identificatie': 'id'})

# Merge residential_heat_demand with export_df on 'id'
merged_df = pd.merge(residential_heat_demand, export_df, on='id', how='inner')

In [5]:
# --- Transmission node definition ---

import warnings
warnings.filterwarnings("ignore", message=".*ChainedAssignmentError: behaviour will change in pandas 3.0!.*")

# Path to the PBF file
pbf_path = "delft.osm.pbf"

# Define the bounding box as a shapely Polygon with four vertices (WGS84 coordinates)
from shapely.geometry import Polygon

# List of (lon, lat) tuples for the four corners (in order, must form a closed ring)
bbox_coords = [
    (4.358862898190234, 51.989950476011565),
    (4.363513483963136, 51.99116041768696),
    (4.359959662710779, 51.997215138653104),
    (4.356868839817735, 51.996580034452485),
    (4.355168730042115, 51.995518297847504),
    (4.35819822166681, 51.9901601698577),
    (4.358862898190234, 51.989950476011565)    # repeat the first point to close the polygon
]
bbox_polygon = Polygon(bbox_coords)

# Load the OSM data with bounding polygon
osm = pyrosm.OSM(pbf_path, bounding_box=bbox_polygon)

# Get all roads and foot paths (including footways, cycleways, etc.)
streets = osm.get_network(network_type="all")

# Ensure geometries are LineStrings or MultiLineStrings
streets = streets[streets.geometry.type.isin(["LineString", "MultiLineString"])]

# Explode MultiLineStrings to LineStrings
streets_exploded = streets.explode(index_parts=False).reset_index(drop=True)

# Collect all start and end points as shapely Points
all_points = []
for geom in streets_exploded.geometry:
    all_points.append(Point(geom.coords[0]))
    all_points.append(Point(geom.coords[-1]))

all_points_gs = gpd.GeoSeries(all_points, crs=streets_exploded.crs)

# Count occurrences of each point (intersections occur more than once)
points_df = all_points_gs.value_counts().reset_index()
points_df.columns = ["geometry", "count"]

# Identify cul-de-sacs (endpoints that appear only once)
culdesacs_df = points_df[points_df["count"] == 1].copy()
culdesacs_df = culdesacs_df.copy()
culdesacs_df.loc[:, "type"] = "culdesac"

# Identify intersections (points that appear more than once)
intersections_df = points_df[points_df["count"] > 1].copy()
intersections_df = intersections_df.copy()
intersections_df.loc[:, "type"] = "intersection"

# Combine intersections and cul-de-sacs
all_nodes_df = pd.concat([intersections_df, culdesacs_df], ignore_index=True)

# Convert to GeoDataFrame
all_nodes_gdf = gpd.GeoDataFrame(all_nodes_df, geometry="geometry", crs=streets_exploded.crs)

# Project to WGS84 if needed
if all_nodes_gdf.crs != "EPSG:4326":
    all_nodes_gdf = all_nodes_gdf.to_crs(epsg=4326)

all_nodes_gdf = all_nodes_gdf.copy()
all_nodes_gdf.loc[:, "lon"] = all_nodes_gdf.geometry.x
all_nodes_gdf.loc[:, "lat"] = all_nodes_gdf.geometry.y

In [6]:
demand_nodes=merged_df
transmission_nodes=all_nodes_gdf

In [7]:
# --- Interpolate between transmission nodes (all_nodes_gdf) ---

# Set the desired spacing in meters between interpolated nodes
spacing_m = 5  # Change this value to control node density

# Use the haversine_distance function already defined in the notebook
# (returns distance in kilometers)
def interpolate_line(lat1, lon1, lat2, lon2, spacing_m=1.0):
    total_dist_km = haversine_distance(lat1, lon1, lat2, lon2)
    total_dist_m = total_dist_km * 1000
    if total_dist_m < 1e-6:
        return [(lat1, lon1), (lat2, lon2)]
    n_points = int(np.floor(total_dist_m / spacing_m))
    if n_points < 1:
        return [(lat1, lon1), (lat2, lon2)]
    lats = np.linspace(lat1, lat2, n_points + 2)
    lons = np.linspace(lon1, lon2, n_points + 2)
    return list(zip(lats, lons))

# Build a list of all unique node pairs (edges) from the street network
edges = []
for idx, street in streets_exploded.iterrows():
    coords = list(street.geometry.coords)
    for i in range(len(coords) - 1):
        pt1 = coords[i]
        pt2 = coords[i+1]
        edges.append((pt1, pt2))

# Map node coordinates to their (rounded) values for matching
node_coords_set = set([(round(pt.x, 6), round(pt.y, 6)) for pt in all_nodes_gdf.geometry])

# Only keep edges where both endpoints are in the node set
filtered_edges = [e for e in edges if (round(e[0][0], 6), round(e[0][1], 6)) in node_coords_set and (round(e[1][0], 6), round(e[1][1], 6)) in node_coords_set]

# Interpolate points for each edge using the global spacing_m
interpolated_points = []
for pt1, pt2 in filtered_edges:
    lat1, lon1 = pt1[1], pt1[0]
    lat2, lon2 = pt2[1], pt2[0]
    points = interpolate_line(lat1, lon1, lat2, lon2, spacing_m=spacing_m)
    interpolated_points.extend(points)

# Remove duplicates and create a GeoDataFrame
unique_points = list({(round(lat, 7), round(lon, 7)) for lat, lon in interpolated_points})
interp_gdf = gpd.GeoDataFrame(geometry=[Point(lon, lat) for lat, lon in unique_points], crs="EPSG:4326")
interp_gdf["lat"] = interp_gdf.geometry.y
interp_gdf["lon"] = interp_gdf.geometry.x

In [8]:
# --- Folium map: interpolated transmission nodes and demand nodes ---

# Center the map on the mean coordinates of all points (demand + interpolated transmission)
center_lat = np.mean(np.concatenate([demand_nodes['lat'].values, interp_gdf['lat'].values]))
center_lon = np.mean(np.concatenate([demand_nodes['lon'].values, interp_gdf['lon'].values]))

map2 = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles="OpenStreetMap")

# Add demand nodes (red)
for _, row in demand_nodes.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.7,
        popup=folium.Popup(f"<b>Demand Node</b><br>ID: {row.get('id', row.get('nodes', ''))}", max_width=250)
    ).add_to(map2)

# Add interpolated transmission nodes (blue)
for _, row in interp_gdf.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=2,
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.5,
        popup=folium.Popup(f"<b>Transmission Node</b><br>Lat: {row['lat']:.6f}<br>Lon: {row['lon']:.6f}", max_width=200)
    ).add_to(map2)

# --- Add lines between interpolated nodes along each street segment ---
# For each street segment, interpolate points and connect each consecutive pair
for idx, street in streets_exploded.iterrows():
    coords = list(street.geometry.coords)
    for i in range(len(coords) - 1):
        pt1 = coords[i]
        pt2 = coords[i+1]
        # Interpolate points for this segment using the same function and spacing_m
        lat1, lon1 = pt1[1], pt1[0]
        lat2, lon2 = pt2[1], pt2[0]
        points = interpolate_line(lat1, lon1, lat2, lon2, spacing_m=spacing_m)
        # Draw lines between each consecutive pair of interpolated points
        for j in range(len(points) - 1):
            start_lat, start_lon = points[j]
            end_lat, end_lon = points[j+1]
            folium.PolyLine(
                locations=[[start_lat, start_lon], [end_lat, end_lon]],
                color="#3186cc",
                weight=2,
                opacity=1
            ).add_to(map2)

# Save and preview
map2.save("intermediary_map.html")

In [ ]:
# --- Transmission node interpolation ---

# 1. Isolate the transmission network and coordinates
transmission_network = pd.read_csv('manual_inputs/transmission_network.csv')
coordinates_df = pd.read_csv('manual_inputs/manual_nodes_base_info.csv')

# 2. Merge coordinates onto the transmission network links
links_with_coords = pd.merge(
    transmission_network,
    coordinates_df[['nodes', 'latitude', 'longitude']],
    left_on='link_from',
    right_on='nodes',
    how='left'
).rename(columns={'latitude': 'lat_from', 'longitude': 'lon_from'})

links_with_coords = pd.merge(
    links_with_coords,
    coordinates_df[['nodes', 'latitude', 'longitude']],
    left_on='link_to',
    right_on='nodes',
    how='left',
    suffixes=('_from_node', '_to_node')
).rename(columns={'latitude': 'lat_to', 'longitude': 'lon_to'})

# 3. Interpolate points for each link and create the new links
interpolated_nodes_list = []
interpolated_links_list = []
num_interpolations = 10

for _, row in links_with_coords.iterrows():
    # --- This part generates the interpolated nodes (same as before) ---
    latitudes = np.linspace(row['lat_from'], row['lat_to'], num=num_interpolations + 2)
    longitudes = np.linspace(row['lon_from'], row['lon_to'], num=num_interpolations + 2)
    
    # Generate node names for the entire chain (start + interp + end)
    node_chain = (
        [row['link_from']] +
        [f"{row['link_from']}_{row['link_to']}_interp_{i+1}" for i in range(num_interpolations)] +
        [row['link_to']]
    )

    # Add the new interpolated nodes to a list for the new nodes DataFrame
    for i in range(num_interpolations):
        interpolated_nodes_list.append({
            'nodes': node_chain[i + 1],
            'latitude': latitudes[i + 1],
            'longitude': longitudes[i + 1],
            'comment': f"Interpolated node for link {row['techs']}"
        })

    # --- This new part generates the links between the nodes in the chain ---
    is_heat = row['techs'].startswith('TH')
    
    for i in range(len(node_chain) - 1):
        link_from_node = node_chain[i]
        link_to_node = node_chain[i+1]
        
        interpolated_links_list.append({
            'techs': f"{link_from_node}_to_{link_to_node}",
            'color': '#823739' if is_heat else '#6783E3',
            'name': 'Interpolated Heat transmission' if is_heat else 'Interpolated Electricity transmission',
            'base_tech': 'transmission',
            'flow_cap_max': '2000',
            'flow_out_eff_per_distance': '0.98' if is_heat else '0.99',
            'lifetime': '20',
            'link_from': link_from_node,
            'link_to': link_to_node
        })

# 4. Create the new DataFrames
new_nodes_df = pd.DataFrame(interpolated_nodes_list)
interpolated_links_df = pd.DataFrame(interpolated_links_list)

interpolated_links_df.to_csv('data_tables/interpolated_transmission_network.csv', index=False)

# 5. Append new nodes to the existing nodes_base_info.csv
# We use concat to combine the old and new data and then save, overwriting the old file.
nodes_base_info_df=pd.read_csv('manual_inputs/manual_nodes_base_info.csv')
updated_nodes_base_info_df = pd.concat([nodes_base_info_df, new_nodes_df], ignore_index=True)
updated_nodes_base_info_df.to_csv('data_tables/nodes_base_info.csv', index=False)

# 6. Update the nodes.csv file to define technologies at each interpolated node
existing_nodes_df = pd.read_csv('manual_inputs/manual_nodes.csv')

nodes_techs_to_add = []
for _, node_row in new_nodes_df.iterrows():
    node_name = node_row['nodes']
    
    # Determine if it's a heat or electricity node based on its name
    is_heat_path = 'TH' in node_name
    is_electricity_path = 'TE' in node_name
    
    # Add demand_heat or demand_electricity with sink_use_equals = 0
    # This makes the node capable of having demand for that carrier, even if it's zero.
    if is_heat_path:
        nodes_techs_to_add.append({
            'nodes': node_name,
            'techs': 'demand_heat',
            'parameters': 'sink_use_equals',
            'timesteps': '',
            '2050/01/01 00:00': 0
        })
    if is_electricity_path:
        nodes_techs_to_add.append({
            'nodes': node_name,
            'techs': 'demand_electricity',
            'parameters': 'sink_use_equals',
            'timesteps': '',
            '2050/01/01 00:00': 0
        })

new_nodes_techs_df = pd.DataFrame(nodes_techs_to_add)

# Concatenate with existing nodes_df and save
updated_nodes_df = pd.concat([existing_nodes_df, new_nodes_techs_df], ignore_index=True)
updated_nodes_df.to_csv('data_tables/nodes.csv', index=False)

# 5. Create Carrier DataFrames for the new links
interpolated_heat_links_techs = interpolated_links_df[interpolated_links_df['name'].str.contains('Heat')]['techs']
interpolated_electricity_links_techs = interpolated_links_df[interpolated_links_df['name'].str.contains('Electricity')]['techs']

interpolated_heat_carrier_df = pd.DataFrame({
    'techs': interpolated_heat_links_techs,
    'carrier_in': 1,
    'carrier_out': 1
})

interpolated_electricity_carrier_df = pd.DataFrame({
    'techs': interpolated_electricity_links_techs,
    'carrier_in': 1,
    'carrier_out': 1
})

# 6. Create Cost DataFrames for the new links
interpolated_heat_costs_df = pd.DataFrame({
    'techs': interpolated_heat_links_techs,
    'cost_flow_cap_per_distance': 100
})

interpolated_electricity_costs_df = pd.DataFrame({
    'techs': interpolated_electricity_links_techs,
    'cost_flow_cap_per_distance': 50
})

In [ ]:
# --- Procedural generation of distribution network ---

demand_nodes=updated_nodes_base_info_df[updated_nodes_base_info_df['nodes'].str.contains('D')].copy()
heat_transmission_nodes=updated_nodes_base_info_df[updated_nodes_base_info_df['nodes'].str.contains('TH')].copy()
electricity_transmission_nodes=updated_nodes_base_info_df[updated_nodes_base_info_df['nodes'].str.contains('TE')].copy()

# Extract coordinates for all three types
demand_coords = demand_nodes[['latitude', 'longitude']].values
heat_transmission_coords = heat_transmission_nodes[['latitude', 'longitude']].values
electricity_transmission_coords = electricity_transmission_nodes[['latitude', 'longitude']].values

# Calculate distances using the Haversine formula
# We pass the haversine function to cdist, which will apply it to every pair of coordinates.
heat_distances = cdist(demand_coords, heat_transmission_coords, 
                       lambda u, v: haversine_distance(u[0], u[1], v[0], v[1]))
electricity_distances = cdist(demand_coords, electricity_transmission_coords, 
                              lambda u, v: haversine_distance(u[0], u[1], v[0], v[1]))

# Find the index of the nearest node for each type separately
nearest_heat_nodes_idx = np.argmin(heat_distances, axis=1)
nearest_electricity_nodes_idx = np.argmin(electricity_distances, axis=1)

# Assign the correct nodes
demand_nodes['heat_node'] = heat_transmission_nodes.iloc[nearest_heat_nodes_idx]['nodes'].values
demand_nodes['electricity_node'] = electricity_transmission_nodes.iloc[nearest_electricity_nodes_idx]['nodes'].values

# Create links dataframe and write to csv
heat_links = pd.DataFrame({
    'techs': demand_nodes['nodes'] + '_to_' + demand_nodes['heat_node'],
    'color': '#823739',
    'name': 'Heat distribution',
    'base_tech': 'transmission',
    'flow_cap_max': '2000',
    'flow_out_eff_per_distance': '0.98',
    'lifetime': '20',
    'link_to': demand_nodes['nodes'],
    'link_from': demand_nodes['heat_node']
}).reset_index(drop=True)

electricity_links=pd.DataFrame({
    'techs': demand_nodes['nodes'] + '_to_' + demand_nodes['electricity_node'],
    'color': '#6783E3',
    'name': 'Electricity distribution',
    'base_tech': 'transmission',
    'flow_cap_max': '2000',
    'flow_out_eff_per_distance': '0.99',
    'lifetime': '20',
    'link_to': demand_nodes['nodes'],
    'link_from': demand_nodes['electricity_node']
}).reset_index(drop=True)

distribution_techs=pd.concat([heat_links, electricity_links], ignore_index=True)

transmission_network=pd.read_csv('manual_inputs/transmission_network.csv')
updated_links = pd.concat([interpolated_links_df, distribution_techs], ignore_index=True)
updated_links.to_csv('data_tables/links.csv', index=False)

# Create carrier dataframes and write to csv
distribution_heat=pd.DataFrame({
    'techs': demand_nodes['nodes'] + '_to_' +  demand_nodes['heat_node'],
    'carrier_out': '1',
    'carrier_in': '1'
    }).reset_index(drop=True)

transmission_heat=pd.read_csv('manual_inputs/transmission_heat.csv')
updated_heat_links=pd.concat([interpolated_heat_carrier_df, distribution_heat], ignore_index=True)
updated_heat_links.to_csv('data_tables/links_heat.csv', index=False)

distribution_electricity=pd.DataFrame({
    'techs': demand_nodes['nodes'] + '_to_' +  demand_nodes['electricity_node'],
    'carrier_out': '1',
    'carrier_in': '1'
    }).reset_index(drop=True)

transmission_electricity=pd.read_csv('manual_inputs/transmission_electricity.csv')
updated_electricity_links=pd.concat([interpolated_electricity_carrier_df, distribution_electricity], ignore_index=True)
updated_electricity_links.to_csv('data_tables/links_electricity.csv', index=False)

# Create costs dataframe and write to csv
distribution_heat_costs=pd.DataFrame({
    'techs': demand_nodes['nodes']  + '_to_' +  demand_nodes['heat_node'],
    'cost_flow_cap_per_distance': '100'
    }).reset_index(drop=True)

distribution_electricity_costs=pd.DataFrame({
    'techs': demand_nodes['nodes']  + '_to_' +  demand_nodes['electricity_node'],
    'cost_flow_cap_per_distance': '50'
    }).reset_index(drop=True)

distribution_costs=pd.concat([distribution_heat_costs, distribution_electricity_costs], ignore_index=True)

transmission_costs=pd.read_csv('manual_inputs/transmission_costs.csv')
updated_links_costs = pd.concat([interpolated_heat_costs_df, interpolated_electricity_costs_df, distribution_costs], ignore_index=True)
updated_links_costs.to_csv('data_tables/links_costs.csv', index=False)


In [ ]:
# --- Scenario creation and model running ---

#calliope.set_log_verbosity("INFO", include_solver_output=True)

# --- Scenario definition ---
# Set the scenario to 'full_electrification' or 'district heating'.
scenario = 'district_heating'

if scenario == 'full_electrification':
    
    # 1. Load the nodes data that was just created in the previous cell
    nodes_base_info_df = pd.read_csv('data_tables/nodes_base_info.csv')
    demand_nodes = nodes_base_info_df[nodes_base_info_df['nodes'].str.startswith('D')]['nodes']

    # 2. Read the model.yaml file
    # Using ruamel.yaml to preserve comments and structure
    yaml = YAML()
    yaml_path = 'district_heating_model.yaml'
    with open(yaml_path, 'r') as f:
        model_config = yaml.load(f)


    
    # 3. Add the heat pump technology to each demand node in the YAML structure
    # Create the top-level 'nodes' key if it doesn't exist
    if 'nodes' not in model_config:
        model_config['nodes'] = {}
    
    # For each demand node, add an entry to allow 'heat_pump' to be built
    for node_name in demand_nodes:
        if node_name not in model_config['nodes']:
            model_config['nodes'][node_name] = {}
        if 'techs' not in model_config['nodes'][node_name]:
            model_config['nodes'][node_name]['techs'] = {}
        # Adding the technology with an empty dictionary is enough to make it available
        model_config['nodes'][node_name]['techs']['heat_pump'] = {}
        
    # 4. Write the updated configuration back to the model.yaml file
    new_yaml_path = 'electrification_model.yaml'
    with open(new_yaml_path, 'w') as f:
        yaml.dump(model_config, f)

    # 5. Deactivate district heating supply node
    nodes_df = pd.read_csv('data_tables/nodes.csv')
    sh_demand_electricity_mask = (nodes_df['nodes'].str.startswith('SH'))
    nodes_df.loc[sh_demand_electricity_mask, '2050/01/01 00:00'] = 0
    nodes_df.to_csv('data_tables/nodes.csv', index=False)
    
    model = calliope.read_yaml("electrification_model.yaml")

elif scenario == 'district_heating':
    model = calliope.read_yaml("district_heating_model.yaml")


In [ ]:
# --- Building and solving of Calliope model ---

model.build()
model.solve()

In [ ]:
# --- Calliope model results visualization ---

df_coords = model.inputs[["latitude", "longitude"]].to_dataframe().reset_index()
df_capacity = (
    model.results.flow_cap.where(model.inputs.base_tech == "transmission")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

# Define distribution and transmission dataframes for plotting
df_capacity_coords = pd.merge(df_coords, df_capacity, left_on="nodes", right_on="nodes").sort_values(by=['techs'])

# Extract link information from techs column (format: "node_from_to_node_to")
df_links = df_capacity_coords.copy()

# Split the techs column to get link_from and link_to
df_links[['link_from', 'link_to']] = df_links['techs'].str.rsplit('_to_', n=1, expand=True)

# Merge with df_coords twice to get both from and to coordinates
# First merge for "from" coordinates
df_links = df_links.merge(
    df_coords[['nodes', 'latitude', 'longitude']],
    left_on='link_from',
    right_on='nodes',
    how='left',
    suffixes=('', '_from')
)
df_links = df_links.rename(columns={'latitude': 'lat_from', 'longitude': 'lon_from'})

# Second merge for "to" coordinates
df_links = df_links.merge(
    df_coords[['nodes', 'latitude', 'longitude']],
    left_on='link_to',
    right_on='nodes',
    how='left',
    suffixes=('_temp', '_to')
)
df_links = df_links.rename(columns={'latitude': 'lat_to', 'longitude': 'lon_to'})

# Clean up duplicate columns
df_links = df_links.drop(columns=['nodes_temp', 'nodes_to'], errors='ignore')


# Create a Folium map centered on your data
center_lat = df_coords['latitude'].mean()
center_lon = df_coords['longitude'].mean()

map_fig = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=16,
    tiles='OpenStreetMap'
)


# Create FeatureGroups for different layers
demand_group = folium.FeatureGroup(name="Demand Nodes", show=True).add_to(map_fig)
supply_heat_group = folium.FeatureGroup(name="Supply Heat Nodes", show=True).add_to(map_fig)
supply_elec_group = folium.FeatureGroup(name="Supply Electricity Nodes", show=True).add_to(map_fig)
transmission_heat_group = folium.FeatureGroup(name="Heat Transmission Nodes", show=True).add_to(map_fig)
heat_link_group = folium.FeatureGroup(name="Heat Links", show=True).add_to(map_fig)
transmission_electricity_group = folium.FeatureGroup(name="Electricity Transmission Nodes", show=True).add_to(map_fig)
electricity_link_group = folium.FeatureGroup(name="Electricity Links", show=True).add_to(map_fig)

# Add lines for each link to the 'link_group' FeatureGroup
for idx, row in df_links.iterrows():
    if row['carriers'] == 'heat':
        color = 'green'
        target_group = heat_link_group
    else:
        color = 'blue'
        target_group = electricity_link_group
        
    folium.PolyLine(
        locations=[[row['lat_from'], row['lon_from']], [row['lat_to'], row['lon_to']]],
        color=color,
        weight=1,
        opacity=0.7,
        popup=f"<b>{row['techs']}</b><br>From: {row['link_from']}<br>To: {row['link_to']}<br>Capacity: {row['Flow capacity (kW)']} kW"
    ).add_to(target_group)


# Add node markers to their respective FeatureGroups
for idx, row in df_capacity_coords.iterrows():
    node_name = row['nodes']
    
    # Determine node type and styling
    if node_name.startswith('SH'):
        color = '#2ecc71' 
        radius = 1
        node_type = 'Supply heat'
        target_group = supply_heat_group
    elif node_name.startswith('SE'):
        color = "#2e38cc" 
        radius = 1
        node_type = 'Supply electricity'
        target_group = supply_elec_group
    elif node_name.startswith('D'):
        color = '#e74c3c'  
        radius = 1
        node_type = 'Demand'
        target_group = demand_group
    elif node_name.startswith('TH'):
        color = "#94d3ae" 
        radius = 1
        node_type = 'Transmission heat'
        target_group = transmission_heat_group
    else:  
        color = "#7076cc"  
        radius = 1
        node_type = 'Transmission electricity'
        target_group = transmission_electricity_group
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=radius,
        popup=f"<b>{row['nodes']}</b> ({node_type})<br>Capacity: {row['Flow capacity (kW)']} kW",
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.8,
        weight=2
    ).add_to(target_group) # Add to the correct group

# --- Add the LayerControl to the map ---
# This creates the toggle switch in the top-right corner
folium.LayerControl().add_to(map_fig)

# Display the map
map_fig.save("outputs/map_output.html")

In [ ]:
# --- Bill of materials export ---

# For each item to be exported, find its name in inputs, merge with capacity data, and export to dataframe
tech_names = model.inputs.name.to_series().dropna()
tech_distances = model.inputs.distance.to_series().dropna()

total_flow_out = (
    model.results.flow_out
    .sum(dim=["nodes", "carriers", "timesteps"], min_count=1)
    .to_series()
    .dropna()
)

export_df = pd.DataFrame({
    'name': tech_names,
    'capacity_kw': total_flow_out,
    'distance_m': tech_distances*1000
})

final_export_df = export_df[export_df['capacity_kw'] > 0].sort_values(by='name')

# For each item in the export dataframe, multiply capacity by some environmental impact factor, and add environmental impact column

environmental_impact_factors = {
    "National grid import": 1,  # e.g. gCO2/kW
    "Geothermal heat extraction": 1,   
    "Heat transmission": 1,             
    "Electricity transmission": 1,      
    "Heat distribution": 1,             
    "Electricity distribution": 1,
    "Air-to-air heat pump": 1,      
}

final_export_df['environmental_impact_per_kW'] = final_export_df['name'].map(environmental_impact_factors)

# Sum total environmental impact across all items and output total environmental impact

final_export_df['environmental_impact'] = final_export_df['capacity_kw'] * final_export_df['environmental_impact_per_kW']
final_export_df = final_export_df.reset_index()

final_export_df.to_csv('outputs/bill_of_materials.csv', index=False)

final_export_df.head()

,techs,name,capacity_kw,distance_m,environmental_impact_per_kW,environmental_impact
0,supply_geothermal,Geothermal heat extraction,1498.297884,NaN,1.0,1498.297884
1,D54_to_TH37_TH36_interp_4,Heat distribution,10.000000,5.713067,1.0,10.000000
2,D55_to_TH36_TH33_interp_7,Heat distribution,10.000000,8.971839,1.0,10.000000
3,D56_to_TH34_TH33_interp_3,Heat distribution,10.000000,10.943926,1.0,10.000000
4,D57_to_TH34,Heat distribution,10.000000,9.489261,1.0,10.000000
